# Demo: GNN-BERT music context

End-to-end inference with trained checkpoints:

1. **Task 1** — BERT caption to multi-label tags (MusicCaps)
2. **Task 2** — GraphSAGE on a chord-transition graph to FMA genre (+ CNN mel baseline)
3. **Task 3** — GNN-BERT fusion to multi-label genre/tags (+ emotion heads)
4. **Task 4** — Contrastive retrieval between captions and audio segment graphs

Run from the repo root or from `notebooks/`.


In [2]:
from __future__ import annotations

import json
import logging
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml

# Quiet noisy library logs that look like failures in notebooks
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
warnings.filterwarnings("ignore", category=UserWarning)
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

try:
    from IPython.display import Image, display
except ImportError:
    def display(x):
        print(x)

    class Image:  # noqa: N801
        def __init__(self, *a, **k):
            self.args = a
            self.kwargs = k

ROOT = Path.cwd()
if not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from torch_geometric.data import Batch

from src.bert_encoder import BertMusicTagClassifier, build_tokenizer, tokenize_batch
from src.contrastive import DualEncoderContrastive
from src.fusion_model import GNNBertFusionModel
from src.gnn_model import CNNMelBaseline, MusicGraphSAGE
from src.graph_builder import build_chord_transition_graph, build_segment_graph

cfg = yaml.safe_load((ROOT / "config.yaml").read_text(encoding="utf-8"))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
max_len = int(cfg["text"]["max_length"])
print(f"ROOT={ROOT}")
print(f"device={device}")


[ok] cell 1


## Results snapshot (`results/metrics.json`)

In [4]:
metrics = json.loads((ROOT / "results" / "metrics.json").read_text(encoding="utf-8"))

rows = []
t1 = metrics["task1_bert_musiccaps"]["test"]
rows.append({"model": "Task 1 BERT tags", "metric": "Macro-F1", "value": t1["macro_f1"]})
rows.append({"model": "Task 1 BERT tags", "metric": "Micro-F1", "value": t1["micro_f1"]})
rows.append({"model": "Task 1 BERT tags", "metric": "AUC-PR", "value": t1["auc_pr"]})

t2 = metrics["task2_fma_small"]
rows.append({"model": "Task 2 majority", "metric": "Accuracy", "value": t2["majority_baseline"]["test_accuracy"]})
rows.append({"model": "Task 2 GNN", "metric": "Accuracy", "value": t2["gnn"]["test"]["accuracy"]})
rows.append({"model": "Task 2 CNN mel", "metric": "Accuracy", "value": t2["cnn_mel_baseline"]["test"]["accuracy"]})

t3 = metrics["task3_gnn_bert_fusion"]["ablations"]
for mode, vals in t3.items():
    rows.append({"model": f"Task 3 {mode}", "metric": "Macro-F1", "value": vals["test"]["macro_f1"]})

t4 = metrics["task4_contrastive_musiccaps"]["test_retrieval"]
for k in ("caption2audio_R@1", "caption2audio_R@5", "caption2audio_R@10",
          "audio2caption_R@1", "audio2caption_R@5", "audio2caption_R@10"):
    rows.append({"model": "Task 4 retrieval", "metric": k, "value": t4[k]})

display(pd.DataFrame(rows).round(3))


[ok] cell 3


## Task 1 — Caption → top tags

Loads `results/checkpoints/task1_bert_best.pt` and predicts MusicCaps aspect tags from a free-text caption.

In [6]:
ckpt1 = torch.load(
    ROOT / "results" / "checkpoints" / "task1_bert_best.pt",
    map_location=device,
    weights_only=False,
)
tag_to_id = ckpt1["tag_to_id"]
id_to_tag = {int(v): k for k, v in tag_to_id.items()}
model1 = BertMusicTagClassifier(
    model_name=ckpt1["config_model_name"],
    num_labels=int(ckpt1["num_labels"]),
).to(device)
model1.load_state_dict(ckpt1["model_state"])
model1.eval()
tok = build_tokenizer(ckpt1["config_model_name"])
max_len = int(cfg["text"]["max_length"])

demo_caption = (
    "This song contains digital drums playing a simple groove along with two guitars. "
    "One strumming chords along with the snare the other one playing a melody on top."
)

with torch.no_grad():
    batch = tokenize_batch([demo_caption], tok, max_len)
    logits = model1(batch["input_ids"].to(device), batch["attention_mask"].to(device))
    probs = torch.sigmoid(logits)[0].cpu().numpy()

top_idx = np.argsort(-probs)[:10]
print("Caption:")
print(demo_caption)
print("\nTop-10 predicted aspects:")
for i in top_idx:
    print(f"  {probs[i]:.3f}  {id_to_tag[int(i)]}")


[ok] cell 5


## Task 2 — Segment graph → genre (GNN vs CNN)

Loads `task2_gnn_best.pt` and `task2_cnn_best.pt` on one FMA-small track: GraphSAGE on the segment graph, CNN on the mel spectrogram (baseline B2).

In [8]:
splits = json.loads((ROOT / "data" / "splits" / "fma_small_splits.json").read_text(encoding="utf-8"))
genre_to_id = splits["genre_to_id"]
id_to_genre = {int(v): k for k, v in genre_to_id.items()}

TRACK_ID = 704
npz_path = ROOT / "data" / "processed" / "fma_small" / f"{TRACK_ID:06d}.npz"
arr = np.load(npz_path)
if str(cfg["graph"].get("type", "chord_transition")) == "chord_transition":
    g = build_chord_transition_graph(arr["chroma"], track_id=TRACK_ID)
else:
    g = build_segment_graph(
        arr["segment_vectors"],
        similarity_threshold=float(cfg["graph"]["similarity_threshold"]),
        temporal_edges=bool(cfg["graph"]["temporal_edges"]),
        similarity_edges=bool(cfg["graph"]["similarity_edges"]),
        track_id=TRACK_ID,
    )
true_genre = next(
    (row["genre"] for split in ("train", "val", "test") for row in splits[split] if int(row["track_id"]) == TRACK_ID),
    None,
)

in_channels = int(g.x.size(1))
gnn2 = MusicGraphSAGE(
    in_channels,
    hidden_channels=int(cfg["model"]["gnn_hidden_dim"]),
    num_layers=int(cfg["model"]["gnn_layers"]),
    num_classes=len(genre_to_id),
    dropout=float(cfg["model"]["gnn_dropout"]),
).to(device)
ckpt_gnn = torch.load(ROOT / "results" / "checkpoints" / "task2_gnn_best.pt", map_location=device, weights_only=False)
gnn2.load_state_dict(ckpt_gnn["model_state"])
gnn2.eval()

cnn2 = CNNMelBaseline(num_classes=len(genre_to_id), n_mels=int(cfg["audio"]["n_mels"])).to(device)
ckpt_cnn = torch.load(ROOT / "results" / "checkpoints" / "task2_cnn_best.pt", map_location=device, weights_only=False)
cnn2.load_state_dict(ckpt_cnn["model_state"])
cnn2.eval()

batch_g = Batch.from_data_list([g]).to(device)
mel = torch.from_numpy(np.asarray(arr["mel"], dtype=np.float32)).unsqueeze(0).to(device)  # (1, n_mels, T)

with torch.no_grad():
    gnn_probs = torch.softmax(gnn2(batch_g.x, batch_g.edge_index, batch_g.batch), dim=-1)[0].cpu().numpy()
    cnn_probs = torch.softmax(cnn2(mel), dim=-1)[0].cpu().numpy()

gnn_pred, cnn_pred = int(gnn_probs.argmax()), int(cnn_probs.argmax())
print(f"track_id={TRACK_ID}  true={true_genre}")
print(f"graph: nodes={g.num_nodes} edges={g.edge_index.size(1)}")
print(f"GNN  pred={id_to_genre[gnn_pred]} (p={gnn_probs[gnn_pred]:.3f})")
print(f"CNN  pred={id_to_genre[cnn_pred]} (p={cnn_probs[cnn_pred]:.3f})")
print("\nTest-set reference (from metrics.json):")
print(f"  majority acc={metrics['task2_fma_small']['majority_baseline']['test_accuracy']:.3f}")
print(f"  GNN      acc={metrics['task2_fma_small']['gnn']['test']['accuracy']:.3f}")
print(f"  CNN mel  acc={metrics['task2_fma_small']['cnn_mel_baseline']['test']['accuracy']:.3f}")


[ok] cell 7


## Task 3 — Graph + text → genre

Builds a segment graph from a processed FMA-small track, encodes metadata text with BERT, and predicts genre with the **cross-attention** fusion checkpoint.

In [10]:
splits = json.loads((ROOT / "data" / "splits" / "fma_small_splits.json").read_text(encoding="utf-8"))
genre_to_id = splits["genre_to_id"]

tracks = pd.read_csv(
    ROOT / "data" / "raw" / "fma" / "fma_metadata" / "tracks.csv",
    index_col=0,
    header=[0, 1],
)
genres_meta = pd.read_csv(ROOT / "data" / "raw" / "fma" / "fma_metadata" / "genres.csv")
genre_id_to_title = {
    int(r.genre_id): str(r.title) for r in genres_meta.itertuples(index=False)
}


def track_text(tid: int) -> str:
    row = tracks.loc[int(tid)]
    title = str(row[("track", "title")])
    artist = str(row[("artist", "name")])
    album = str(row[("album", "title")])
    atags = str(row[("artist", "tags")])
    parts = [
        title if title != "nan" else "",
        f"Artist: {artist}" if artist != "nan" else "",
        f"Album: {album}" if album != "nan" else "",
        f"Tags: {atags}" if atags not in {"nan", "[]", ""} else "",
    ]
    return ". ".join(p for p in parts if p) or "unknown track"

TRACK_ID = 704
npz_path = ROOT / "data" / "processed" / "fma_small" / f"{TRACK_ID:06d}.npz"
arr = np.load(npz_path)
gtype = str(cfg["graph"].get("type", "chord_transition"))
if gtype == "chord_transition":
    g = build_chord_transition_graph(arr["chroma"], track_id=TRACK_ID)
else:
    g = build_segment_graph(
        arr["segment_vectors"],
        similarity_threshold=float(cfg["graph"]["similarity_threshold"]),
        temporal_edges=bool(cfg["graph"]["temporal_edges"]),
        similarity_edges=bool(cfg["graph"]["similarity_edges"]),
        track_id=TRACK_ID,
    )
text = track_text(TRACK_ID)
true_genre = next(
    (row["genre"] for split in ("train", "val", "test") for row in splits[split] if int(row["track_id"]) == TRACK_ID),
    None,
)

ckpt3 = torch.load(
    ROOT / "results" / "checkpoints" / "task3_cross_attention_best.pt",
    map_location=device,
    weights_only=False,
)
num_labels = int(ckpt3.get("num_labels", len(genre_to_id)))
model3 = GNNBertFusionModel(
    in_channels=int(g.x.size(1)),
    num_labels=num_labels,
    bert_name=str(cfg["model"]["bert_name"]),
    fusion="cross_attention",
    gnn_type=str(cfg["model"]["gnn_type"]),
    gnn_hidden=int(cfg["model"]["gnn_hidden_dim"]),
    gnn_layers=int(cfg["model"]["gnn_layers"]),
    gnn_dropout=float(cfg["model"]["gnn_dropout"]),
    predict_emotion=True,
).to(device)
model3.load_state_dict(ckpt3["model_state"])
model3.eval()

tok3 = build_tokenizer(str(cfg["model"]["bert_name"]))
batch_g = Batch.from_data_list([g]).to(device)
tok_batch = tokenize_batch([text], tok3, max_len)
with torch.no_grad():
    logits, v_hat, a_hat = model3(
        batch_g.x,
        batch_g.edge_index,
        batch_g.batch,
        tok_batch["input_ids"].to(device),
        tok_batch["attention_mask"].to(device),
        return_emotion=True,
    )
    probs = torch.sigmoid(logits)[0].cpu().numpy()

print(f"track_id={TRACK_ID} graph={gtype} nodes={g.num_nodes} edges={g.edge_index.size(1)}")
print(f"text: {text}")
print(f"true genre (FMA top): {true_genre}")
print(f"pred valence/arousal (heads): {float(v_hat):.3f} / {float(a_hat):.3f}")

vocab = json.loads(
    (ROOT / "data" / "splits" / "fma_small_multilabel_vocab.json").read_text(encoding="utf-8")
)
id_to_name = {}
for gi, gid in enumerate(vocab.get("genre_ids", [])):
    title = genre_id_to_title.get(int(gid), str(gid))
    id_to_name[gi] = f"genre:{title}"
for tag, ti in vocab.get("tag_to_idx", {}).items():
    id_to_name[int(vocab["n_genre"]) + int(ti)] = f"tag:{tag}"

print("Top-10 multi-label scores:")
for i in np.argsort(-probs)[:10]:
    print(f"  {probs[i]:.3f}  {id_to_name.get(int(i), f'label_{int(i)}')}")


[ok] cell 9


## Task 4 — Caption → audio retrieval

Embeds a MusicCaps caption and a small gallery of audio graphs with the dual encoder, then shows top-3 matches. Also prints the saved qualitative examples from training.

In [12]:
mc_csv = pd.read_csv(ROOT / "data" / "raw" / "musiccaps" / "musiccaps-public.csv")
audio_dir = ROOT / "data" / "raw" / "musiccaps" / "audio"
proc_dir = ROOT / "data" / "processed" / "musiccaps"

pairs = []
for _, row in mc_csv.iterrows():
    ytid = str(row["ytid"])
    stem = f"{ytid}_{int(row['start_s'])}_{int(row['end_s'])}"
    wav = audio_dir / f"{ytid}.wav"
    npz = proc_dir / f"{stem}.npz"
    if not wav.exists() or not npz.exists():
        continue
    if not bool(row["is_audioset_eval"]):
        continue
    arr = np.load(npz)
    graph = build_segment_graph(
        arr["segment_vectors"],
        similarity_threshold=float(cfg["graph"]["similarity_threshold"]),
        temporal_edges=bool(cfg["graph"]["temporal_edges"]),
        similarity_edges=bool(cfg["graph"]["similarity_edges"]),
    )
    pairs.append({"stem": stem, "caption": str(row["caption"]), "graph": graph})
    if len(pairs) >= 64:
        break

print(f"gallery size={len(pairs)} (eval clips with audio+features)")
in_ch = int(pairs[0]["graph"].x.size(1))
model4 = DualEncoderContrastive(
    in_channels=in_ch,
    bert_name=str(cfg["model"]["bert_name"]),
    gnn_type=str(cfg["model"]["gnn_type"]),
    gnn_hidden=int(cfg["model"]["gnn_hidden_dim"]),
    gnn_layers=int(cfg["model"]["gnn_layers"]),
    gnn_dropout=float(cfg["model"]["gnn_dropout"]),
    projection_dim=int(cfg["model"]["projection_dim"]),
    temperature=float(cfg["train"]["temperature"]),
).to(device)
ckpt4 = torch.load(
    ROOT / "results" / "checkpoints" / "task4_contrastive_best.pt",
    map_location=device,
    weights_only=False,
)
model4.load_state_dict(ckpt4["model_state"])
model4.eval()
tok4 = build_tokenizer(str(cfg["model"]["bert_name"]))

with torch.no_grad():
    g_batch = Batch.from_data_list([p["graph"] for p in pairs]).to(device)
    t_batch = tokenize_batch([p["caption"] for p in pairs], tok4, max_len)
    g_emb, t_emb = model4(
        g_batch.x,
        g_batch.edge_index,
        g_batch.batch,
        t_batch["input_ids"].to(device),
        t_batch["attention_mask"].to(device),
    )
    sim = (t_emb @ g_emb.t()).cpu()

query_i = 0
top3 = sim[query_i].topk(3).indices.tolist()
print("\nQuery caption:")
print(pairs[query_i]["caption"][:300])
print("\nTop-3 retrieved clips:")
for rank, j in enumerate(top3, 1):
    hit = "OK" if j == query_i else "--"
    print(f"  #{rank} [{hit}] score={sim[query_i, j]:.3f} stem={pairs[j]['stem']}")
    print(f"      {pairs[j]['caption'][:160]}...")


[ok] cell 11


In [13]:
examples_path = ROOT / "results" / "retrieval_examples" / "task4_caption_to_audio_examples.json"
examples = json.loads(examples_path.read_text(encoding="utf-8"))
print(f"Saved qualitative examples: {len(examples)} (from full test eval)\n")
for i, ex in enumerate(examples[:3], 1):
    print(f"=== Example {i} ===")
    print("Q:", ex["query_caption"][:220])
    for m in ex["top3_matched_clips"]:
        mark = "HIT" if m["is_correct"] else "   "
        print(f"  [{mark}] {m['score']:.3f}  {m['caption'][:120]}")
    print()


[ok] cell 12


## Case studies (Task 3)

From `results/task3_case_studies.json` — graph size + caption/metadata alignment notes.

In [15]:
cases = json.loads((ROOT / "results" / "task3_case_studies.json").read_text(encoding="utf-8"))
display(pd.DataFrame(cases))


[ok] cell 14


## Plots

Saved under `results/plots/`:

- `task1_f1_curves.png`
- `task2_gnn_vs_cnn.png`
- `task3_ablation_macro_f1.png`
- `task3_tsne_genre.png`
- `task3_tsne_mood.png`
- `task4_retrieval_r_at_k.png`


In [17]:
plot_names = [
    "task1_f1_curves.png",
    "task2_gnn_vs_cnn.png",
    "task3_ablation_macro_f1.png",
    "task3_tsne_genre.png",
    "task3_tsne_mood.png",
    "task4_retrieval_r_at_k.png",
]
for name in plot_names:
    path = ROOT / "results" / "plots" / name
    if not path.exists():
        print(f"{name}: missing")
        continue
    print(name)
    display(Image(filename=str(path), width=480))


[ok] cell 16
